## A Clothing Company

A **Clothing** Company can produce **shirts, shorts, and pants**.
Each unit requires labor hours and cloth and yields revenue with a variable unit cost. To produce any item type in a week, the firm must rent the corresponding machine (fixed weekly cost). Weekly resources are limited to 150 labor hours and 160 sq yd of cloth. Decide weekly production to maximize profit (revenue − variable costs − machine rentals).

**Data**

| Clothing | Labor (hr/unit) | Cloth (sq yd/unit) | Price (\$/unit) | Variable Cost (\$/unit) | Machine Rental (\$/wk) |
|----------|----------------:|-------------------:|---------------:|-----------------------:|----------------------:|
| Shirt    |               3 |                  4 |             12 |                      6 |                   200 |
| Shorts   |               2 |                  3 |              8 |                      4 |                   150 |
| Pants    |               6 |                  4 |             15 |                      8 |                   100 |


Available per week: **150 labor hours, 160 sq yd cloth**.

In [1]:
pip install gurobipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
from gurobipy import Model, GRB, quicksum

**Notation for A Clothing Company**

\begin{aligned}
&a_i
&& \text{labor hours required per unit of clothing type $i$} \\[6pt]
&b_i
&& \text{cloth (sq yd) required per unit of clothing type $i$} \\[6pt]
&r_i
&& \text{sales price per unit of clothing type $i$} \\[6pt]
&c_i
&& \text{variable cost per unit of clothing type $i$} \\[6pt]
&f_i
&& \text{fixed (rental) cost of machinery for clothing type $i$} \\[6pt]
&L = 150
&& \text{weekly labor hours available} \\[6pt]
&C = 160
&& \text{weekly cloth (sq yd) available} \\[6pt]
&p_i = r_i - c_i
&& \text{unit profit contribution for clothing type $i$} \\[6pt]
&M_i
&& \text{upper bound on units of type $i$ that can be produced under capacities}
\end{aligned}



In [3]:
# Data
types = ["shirt", "shorts", "pants"]

labor_req = {"shirt": 3, "shorts": 2, "pants": 6}   # hours/unit
cloth_req = {"shirt": 4, "shorts": 3, "pants": 4}   # sq yd/unit

labor_cap = 150
cloth_cap = 160

price      = {"shirt": 12, "shorts": 8, "pants": 15}
var_cost   = {"shirt":  6, "shorts": 4, "pants":  8}
unit_profit = {t: price[t] - var_cost[t] for t in types}  # $/unit

# weekly machine rental (fixed) if we produce any of that type
fixed_cost = {"shirt": 200, "shorts": 150, "pants": 100}

# define big-M
M = {t: 10000 for t in types}

# Conservative big-M: max units of each type feasible under resource caps
# M = {
#     t: min(labor_cap // labor_req[t], cloth_cap // cloth_req[t])
#     for t in types
# }

**Decision Variables**

\begin{aligned}
&x_s, x_{sh}, x_p \in \mathbb{Z}_{\ge 0}
&& \text{units of shirts, shorts, and pants produced} \\[6pt]
&y_s, y_{sh}, y_p \in \{0,1\}
&& \text{1 if the corresponding machine is rented, 0 otherwise} \\[6pt]
\end{aligned}

In [4]:
# Model
m = Model("A Clothing Company")

# Decision variables
x = m.addVars(types, vtype=GRB.INTEGER, lb=0, name="make")      # units produced
y = m.addVars(types, vtype=GRB.BINARY,  name="rent_machine")     # 1 if rent machine

Restricted license - for non-production use only - expires 2027-11-29


**Objective Function**

\begin{aligned}
\max \quad & \sum_{i \in \{\text{shirt, shorts, pants}\}} p_i x_i \;-\; \sum_{i} f_i y_i \\
\end{aligned}

In [5]:
# Objective: maximize profit = unit profits – fixed rental costs
m.setObjective(
    quicksum(unit_profit[t] * x[t] for t in types)
    - quicksum(fixed_cost[t] * y[t] for t in types),
    GRB.MAXIMIZE
)

**Constraints**
\begin{aligned}
\text{s.t.}\quad
& \sum_i a_i x_i \le L \quad (\text{labor})\\
& \sum_i b_i x_i \le C \quad (\text{cloth})\\
& x_i \le M_i\, y_i \quad \forall i \quad (\text{linking})\\
& x_i \in \mathbb{Z}_{\ge 0},\; y_i \in \{0,1\} \quad \forall i
\end{aligned}

In [6]:
# Resource constraints
m.addConstr(quicksum(labor_req[t] * x[t] for t in types) <= labor_cap, name="labor")
m.addConstr(quicksum(cloth_req[t] * x[t] for t in types) <= cloth_cap, name="cloth")

# Linking constraints: produce only if the machine is rented
for t in types:
    m.addConstr(x[t] <= M[t] * y[t], name=f"link_{t}")

### Integer Program for A Clothing Company

\begin{aligned}
\max \quad & \sum_{i \in \{\text{shirt, shorts, pants}\}} p_i x_i \;-\; \sum_{i} f_i y_i \\
\text{s.t.}\quad
& \sum_i a_i x_i \le L \quad (\text{labor})\\
& \sum_i b_i x_i \le C \quad (\text{cloth})\\
& x_i \le M_i\, y_i \quad \forall i \quad (\text{linking})\\
& x_i \in \mathbb{Z}_{\ge 0},\; y_i \in \{0,1\} \quad \forall i
\end{aligned}



In [7]:
m.optimize()

# Report
if m.status == GRB.OPTIMAL:
    print(f"Optimal weekly profit: ${m.objVal:.2f}\n")
    for t in types:
        print(f"{t.capitalize():<6}  make = {int(x[t].X):>3} units   rent? = {int(y[t].X)}")
else:    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 5 rows, 6 columns and 12 nonzeros (Max)
Model fingerprint: 0xaae3f618
Model has 6 linear objective coefficients
Variable types: 0 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [4e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+02, 2e+02]

Found heuristic solution: objective -0.0000000
Found heuristic solution: objective 40.0000000
Presolve time: 0.00s
Presolved: 5 rows, 6 columns, 12 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 8.111321e+01, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   81

## Gurobi IP Output Notes

### Model characteristics
- 6 integer variables (3 binary)

### Presolve simplification
- Gurobi simplifies the model before solving  

### Model analysis
- Continuous range of values appears across:
  - Constraint coefficients  
  - Objective coefficients  
  - Upper–lower bounds on decision variables  

### Relaxation and initial solution
- Linear relaxation of the problem solved (root relaxation)
- An initial feasible solution was found at the root node using simplex (4 iterations)  
- Branch-and-bound tree with heuristics applied

### Cutting planes
- Additional constraints added to tighten the formulation  
- Types included:
  - Flow cover

### Search process
- Explored 1 branch-and-bound nodes  
- Performed 4 simplex iterations  

### Solutions found
- Optimal solution identified
